<a href="https://colab.research.google.com/github/Deangr-econ/Quant_methods_project/blob/main/Group_Project_QTFE_Functions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import adfuller, kpss
import statsmodels.api as sm
import scipy.stats as stats
import matplotlib.pyplot as plt

In [ ]:
# --- 1. Garman-Klass Realized SD Function ---
def garman_klass_sd(df, ticker):
    o = df["Open"][ticker] if isinstance(df.columns, pd.MultiIndex) else df["Open"]
    h = df["High"][ticker] if isinstance(df.columns, pd.MultiIndex) else df["High"]
    l = df["Low"][ticker] if isinstance(df.columns, pd.MultiIndex) else df["Low"]
    c = df["Close"][ticker] if isinstance(df.columns, pd.MultiIndex) else df["Close"]

    log_hl = np.log(h / l)
    log_co = np.log(c / o)

    # Daily Garman-Klass variance
    gk_var = 0.5 * (log_hl**2) - (2 * np.log(2) - 1) * (log_co**2)

    # Return daily standard deviation (non-annualized)
    return np.sqrt(np.maximum(gk_var, 0))

In [ ]:
def test_stationarity_table(df, regression="c", alpha=0.05):
    """Runs ADF and KPSS tests on every column of a DataFrame and returns

    a clean, publication-ready summary table.
    """
    records = []

    for col in df.columns:
        series = df[col].dropna()

        # 1. ADF Test (H0: Series has a unit root / is non-stationary)
        adf_res = adfuller(series, regression=regression, result_object=True)
        adf_stat = adf_res.statistic
        adf_p = adf_res.pvalue
        adf_reject = adf_p < alpha  # True -> Stationary

        # 2. KPSS Test (H0: Series is stationary)
        kpss_res = kpss(
            series, regression=regression, nlags="auto", result_object=True
        )
        kpss_stat = kpss_res.statistic
        kpss_p = kpss_res.pvalue
        kpss_reject = kpss_p < alpha  # True -> Non-stationary

        # 3. Combined Inference
        if adf_reject and not kpss_reject:
            verdict = "Stationary"
        elif not adf_reject and kpss_reject:
            verdict = "Non-Stationary (Unit Root)"
        elif not adf_reject and not kpss_reject:
            verdict = "Inconclusive"
        else:
            verdict = "Contradictory / Structural Break"

        records.append({
            "Variable": col,
            "ADF Stat": round(adf_stat, 4),
            "ADF p-value": f"{adf_p:.4e}" if adf_p < 0.001 else round(adf_p, 4),
            "KPSS Stat": round(kpss_stat, 4),
            "KPSS p-value": (
                f"{kpss_p:.4e}" if kpss_p < 0.001 else round(kpss_p, 4)
            ),
            "Conclusion (α=0.05)": verdict,
        })

    summary_df = pd.DataFrame(records).set_index("Variable")
    return summary_df

In [ ]:
def compute_girf_matrix(results, periods=10):
    # Ensure sigma_u is a raw numpy ndarray, not a pandas DataFrame
    sigma_u = np.asarray(results.sigma_u)  # <--- FIX HERE
    names = list(results.names)
    k = len(names)

    # MA coefficient matrices: (periods + 1, K, K)
    ma_rep = results.ma_rep(maxn=periods)

    girf_dict = {
        imp: {resp: np.zeros(periods + 1) for resp in names} for imp in names
    }

    for j, imp_var in enumerate(names):
        sigma_jj = sigma_u[j, j]  # Now standard numpy integer index lookup works
        scale = 1.0 / np.sqrt(sigma_jj)

        cov_vector = sigma_u[:, j]

        for h in range(periods + 1):
            phi_h = ma_rep[h]
            response_at_h = (phi_h @ cov_vector) * scale

            for i, resp_var in enumerate(names):
                girf_dict[imp_var][resp_var][h] = response_at_h[i]

    return girf_dict

In [ ]:
def plot_var_residual_diagnostics(var_results, equation_name, lags=20, bins=70):
    """
    Plots a 4-panel residual diagnostic grid for a single equation in a fitted VAR model.
    """
    # 1. Extract 1D residual series for the target equation
    resid = var_results.resid[equation_name].dropna()

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # --- Panel 1: Residuals Over Time ---
    axes[0, 0].plot(resid.index, resid.values, color="#1f77b4", lw=0.8, alpha=0.85)
    axes[0, 0].axhline(0, color="black", linestyle="--", linewidth=0.9)
    axes[0, 0].set_title(f"Residuals Over Time: {equation_name} (Volatility Clustering)", fontweight="bold")
    axes[0, 0].set_ylabel("Residual")
    axes[0, 0].grid(True, alpha=0.3)

    # --- Panel 2: Density vs Theoretical Normal ---
    axes[0, 1].hist(resid, bins=bins, density=True, alpha=0.55, color="#2ca02c", edgecolor="none")
    mu, std = stats.norm.fit(resid)
    xmin, xmax = axes[0, 1].get_xlim()
    x_pdf = np.linspace(xmin, xmax, 500)
    kurt = stats.kurtosis(resid, fisher=False)  # Pearson kurtosis (normal = 3.0)
    skew = stats.skew(resid)

    axes[0, 1].plot(
        x_pdf, stats.norm.pdf(x_pdf, mu, std), "r--", lw=2,
        label=rf"Normal ($\mu={mu:.3f}, \sigma={std:.3f}$)"
    )
    axes[0, 1].set_title(f"Empirical vs. Normal (Kurtosis: {kurt:.1f}, Skew: {skew:.2f})", fontweight="bold")
    axes[0, 1].set_ylabel("Density")
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    # --- Panel 3: Q-Q Plot (Heavy Tails) ---
    sm.qqplot(resid, line="45", fit=True, ax=axes[1, 0], color="#1f77b4", alpha=0.5)
    axes[1, 0].set_title("Normal Q-Q Plot (Tail Extremity)", fontweight="bold")
    axes[1, 0].grid(True, alpha=0.3)

    # --- Panel 4: Autocorrelation Function (ACF) ---
    sm.graphics.tsa.plot_acf(resid, lags=lags, ax=axes[1, 1], alpha=0.05)
    axes[1, 1].set_title(f"Residual ACF (Up to {lags} Lags)", fontweight="bold")
    axes[1, 1].set_xlabel("Lag")
    axes[1, 1].set_ylabel("Autocorrelation")
    axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

# Example Usage:
# plot_var_residual_diagnostics(results, 'OIL')
# plot_var_residual_diagnostics(results, 'GAS')